# 5. Prompt Engineering

Notes for this chapter:

- 

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-5-5"

In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, stream=False):
    params = {
        "model": model,
        "max_tokens": 16000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    if stream:
        with client.messages.stream(**params) as response:
            for text in response.text_stream:
                print(text, end="", flush=True)
            message = response.get_final_message()
    else:
        message = client.messages.create(**params)

    # Responses can start with a thinking block, so find the text block
    return next(block.text for block in message.content if block.type == "text")

In [3]:
messages = []

add_user_message(messages, "Hello!")
answer = chat(messages)
add_assistant_message(messages, answer)

answer

'Hello! How can I help you today?'

## Exercises

## Dataset setup

In [3]:
from anthropic_building_with_claude.prompt_evaluator import PromptEvaluator

evaluator = PromptEvaluator(max_concurrent_tasks=3)



In [ ]:
dataset = evaluator.generate_dataset(
    task_description="Write a compact, concise 1 day meal plan for a single athlete",
    prompt_inputs_spec={
        "height": "Athlete's height in cm",
        "weight": "Athlete's weight in kg", 
        "goal": "Goal of the athlete",
        "restrictions": "Dietary restrictions of the athlete"
    },
    output_file="dataset.json",
    num_cases=3
)

A first run, producing bad results. This averages ~3/10 across all tests

In [5]:


def run_prompt(prompt_inputs):
    prompt = f"""
What should this person eat?

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown  
- Meals with exact foods, portions, and timing
"""
)


Test 1: Typical male endurance athlete aiming to build muscle with no dietary restrictions (baseline case).
Score: 3/10
Reasoning: The output has a calorie total and macro breakdown and a sensible meal structure with adequate protein. It fails the extra criterion of exact foods, portions and timing, because many portions are missing, options are left open, and timing is only loosely given. It also ignores the compact, concise requirement by adding long extra sections, and the meal calories do not add up to the stated target. Because the extra criteria are missed, the score is capped at 3 or lower.
Strengths:
  + Includes daily calorie target (~2,900-3,000 kcal) and a full macro breakdown, with protein at about 2 g/kg, which is appropriate for muscle gain.
  + Covers breakfast, lunch, dinner, pre- and post-workout meals and an evening snack, and the food choices fit the lean-muscle plus endurance goal.
Weaknesses:
  - Meals lack exact foods, portions and timing: the oatmeal, milk, banan

## We can do better!

Be direct and concise. 

Being "clear" means:

Use simple language that anyone can understand
State exactly what you want without beating around the bush
Lead with a straightforward statement of Claude's task

Being "direct" focuses on how you structure your request:

Use instructions, not questions
Start with direct action verbs like "Write," "Create," or "Generate"

In [6]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown  
- Meals with exact foods, portions, and timing
"""
)

Test 1: Typical male endurance athlete aiming to build muscle with no dietary restrictions (baseline case).
Score: 7/10
Reasoning: The plan covers the required content: breakfast, lunch, dinner, several snacks and a pre/post-workout meal, with calories, macros, portions and timing. The numbers add up reasonably well. Flaws are the truncated final tip, the length for a 'compact' request, and a protein target slightly above the suggested range and not exactly matching the meal totals. These are minor and do not break the core task.
Strengths:
  + Meets all extra criteria: gives a daily calorie total (~3,000 kcal), a macro table, and meals with exact foods, gram portions, and times.
  + Meal macros and calories are internally consistent and sum to about 3,030 kcal, 373 g carbs and 80 g fat, which fits the muscle-gain-with-endurance goal. Pre- and post-workout meals are included.
  + Well structured and easy to scan, with a clear assumptions line and a per-meal macro summary.
Weaknesses:
 

## Another step up - Be Specific!

Add guidelines to shape how the LLM works. There are two main approaches:

### Output Quality Guidelines

The first type focuses on listing qualities that your output should have. These guidelines help you control:

Length of the response
Structure and format
Specific attributes or elements to include
Tone or style requirements

For example, you might specify that a story should be under 1,000 words, include a clear action that reveals the character's talent, and feature at least one supporting character.

### Process Steps

The second type provides specific steps for Claude to follow. This approach is particularly useful when you want Claude to think through a problem systematically or consider multiple perspectives before arriving at a final answer.

Instead of jumping straight to writing, you might ask Claude to:

Brainstorm three talents that would create dramatic tension
Pick the most interesting talent
Outline a pivotal scene that reveals the talent
Brainstorm supporting character types that could increase the impact


### When to use each:

Here's a practical guide for when to use each type of specificity:

**Always Use Output Guidelines**

You should include quality guidelines in almost every prompt you write. They're your safety net for getting consistent, useful results.

**Use Process Steps For Complex Problems**

Add step-by-step instructions when you're dealing with:

Troubleshooting complex problems
Decision-making scenarios
Critical thinking tasks
Any situation where you want Claude to consider multiple angles



In [6]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts  
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""
    
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown  
- Meals with exact foods, portions, and timing
"""
)

Test 1: Typical male endurance athlete aiming to build muscle with no dietary restrictions (baseline case).
Score: 7/10
Reasoning: The output covers all required content, including calories, macros, foods, portions and timing, and the figures are accurate and consistent. Protein runs slightly above the suggested range. The main flaw is conciseness: the repeated totals table and extra notes make it longer than a compact one-day plan needs to be.
Strengths:
  + Meets every extra criterion: gives a daily calorie total (~3,100 kcal), a macro breakdown, and meals with exact foods, gram portions and clock times.
  + Numbers are internally consistent and realistic: per-meal macros sum correctly to the daily totals, and spot checks of foods like oats, eggs and chicken with rice match standard values.
  + The plan fits the goal: carbs are high around training for endurance, protein is spread across six eating occasions, and there is a pre-workout snack, a post-workout dinner and a casein evenin

## Structure with XML tags

Group related information with XML tags when working with large amounts of data to avoid confusion.

This might not provide much improvement to a simply prompt as below, but pays off for larger prompts. 

Tags don't need to be valid html tags, it's better to use tags that describe the data.

A better example is in the model_grader prompt (`grade_output`) in the `PromptEvaluator` itself


In [7]:
def run_prompt(prompt_inputs):
    prompt = f"""
Generate a one-day meal plan for an athlete that meets their dietary restrictions.

<athlete_information>
- Height: {prompt_inputs["height"]}
- Weight: {prompt_inputs["weight"]}
- Goal: {prompt_inputs["goal"]}
- Dietary restrictions: {prompt_inputs["restrictions"]}
</athlete_information>

Guidelines:
1. Include accurate daily calorie amount
2. Show protein, fat, and carb amounts  
3. Specify when to eat each meal
4. Use only foods that fit restrictions
5. List all portion sizes in grams
6. Keep budget-friendly if mentioned
"""
    
    messages = []
    add_user_message(messages, prompt)
    return chat(messages)

results = evaluator.run_evaluation(
    run_prompt_function=run_prompt,
    dataset_file="dataset.json",
    extra_criteria="""
The output should include:
- Daily caloric total
- Macronutrient breakdown  
- Meals with exact foods, portions, and timing
"""
)

Test 1: Typical male endurance athlete aiming to build muscle with no dietary restrictions (baseline case).
Score: 6/10
Reasoning: The plan is complete and numerically consistent, with timing, portions, calories and macros all present, so the extra criteria are satisfied. Its main faults are length and redundancy, which conflict with the compact-and-concise requirement, and protein that overshoots the stated target range. It is useful and accurate but not compact, so it lands in the upper-middle range.
Strengths:
  + Meets every extra criterion: daily calorie total (~3,054 kcal), a macro breakdown, and meals with exact foods, gram portions and clock times. The per-meal figures sum correctly to the stated totals.
  + Covers breakfast, lunch, dinner, a mid-morning snack, and pre- and post-workout meals. The plan is sensible for lean gain with endurance work, with carbs timed around training.
Weaknesses:
  - Not compact or concise. It repeats the same numbers in per-meal lines and again i

# Providing examples

Providing examples in your prompts is one of the most effective prompt engineering techniques you'll use. This approach, known as "one-shot" or "multi-shot" prompting, involves giving Claude sample input/output pairs to guide its responses.

These can be set up with XML tagles:


```text
<sample_input>
Some example case
</sample_input>
<ideal_output>
What we should expect
</ideal_output>

Also explain why this output is ideal

```


## When to Use Examples

Examples are particularly useful for:

Capturing corner cases or edge scenarios
Defining complex output formats (like specific JSON structures)
Showing the exact style or tone you want
Demonstrating how to handle ambiguous inputs

## One-Shot vs Multi-Shot

One-Shot: Provide a single example to establish the pattern

Multi-Shot: Provide multiple examples to cover different scenarios

Use multi-shot when you need to handle various edge cases or want to show different types of valid responses.


# Best Practices

- Always use XML tags to structure your examples clearly
- Be explicit about what you're showing: "Here is an example input with an ideal response"
- Include examples that address your most common failure cases
- Explain why your example outputs are considered ideal
- Keep examples relevant to your specific task